<a href="https://colab.research.google.com/github/Dipu1764/Data-analyst-internship-task-6/blob/DATA/Sales_Trend_Analysis_data_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

# Load dataset
file_path = "/content/online_sales_dataset.csv"  # change path if needed
df = pd.read_csv(file_path)

# 1. Remove rows with negative Quantity or UnitPrice
df_clean = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)].copy()

# 2. Cap Discount between 0 and 1
df_clean['Discount'] = df_clean['Discount'].clip(lower=0, upper=1)

# 3. Standardize text columns
df_clean['PaymentMethod'] = df_clean['PaymentMethod'].str.strip().str.title()
df_clean['PaymentMethod'] = df_clean['PaymentMethod'].replace({'Paypall': 'PayPal'})

df_clean['Country'] = df_clean['Country'].str.strip().str.title()

# 4. Fill missing ShippingCost with median
shipping_median = df_clean['ShippingCost'].median()
df_clean['ShippingCost'] = df_clean['ShippingCost'].fillna(shipping_median)

# 5. Convert InvoiceDate to datetime and extract only the date
df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'], errors='coerce')
df_clean['order_date'] = df_clean['InvoiceDate'].dt.date

# 6. Create amount column
df_clean['amount'] = (
    df_clean['Quantity'] * df_clean['UnitPrice'] * (1 - df_clean['Discount'])
) + df_clean['ShippingCost']

# 7. Rename columns to match SQL schema
df_clean.rename(columns={
    'InvoiceNo': 'order_id',
    'StockCode': 'product_id'
}, inplace=True)

# 8. Keep only relevant columns for SQL project
df_sql = df_clean[['order_id', 'order_date', 'amount', 'product_id']]

# Save cleaned dataset for SQL import
df_sql.to_csv("online_sales_clean.csv", index=False)

print("Data cleaned and saved to online_sales_clean.csv")
print(df_sql.head())


Data cleaned and saved to online_sales_clean.csv
   order_id  order_date     amount product_id
0    221958  2020-01-01    45.2294   SKU_1964
1    771155  2020-01-01   610.9350   SKU_1241
2    231932  2020-01-01   950.1835   SKU_1501
3    465838  2020-01-01   934.3072   SKU_1760
5    744167  2020-01-01  1728.6904   SKU_1006
